# LLM-as-a-Judge Evaluation

This notebook demonstrates using LLMs to evaluate open-ended responses with multi-dimensional scoring.

In [ ]:
import os
import json

from metaeval.judges import create_judge, list_providers, get_default_model, JudgeSettings
from metaeval.prompts import get_registry, get_prompt

## Available Providers

Metaeval supports multiple LLM providers for judging:

In [ ]:
# List available providers
providers = list_providers()
print("Available Providers:")
for provider in providers:
    default_model = get_default_model(provider)
    print(f"  - {provider}: {default_model}")

## Judge Prompts

Different prompt styles for judging:

In [ ]:
# List available judge prompts
registry = get_registry()
judge_prompts = registry.list('judge')

print("Available Judge Prompts:")
for prompt in judge_prompts:
    desc = prompt.description.split('\n')[0][:60] if prompt.description else ''
    print(f"  - {prompt.name}: {desc}")

In [ ]:
# View the multi-dimensional prompt
md_prompt = get_prompt('multi_dimensional', 'judge')
if md_prompt:
    print(f"Prompt: {md_prompt.name}")
    print(f"Variables: {md_prompt.variables}")
    print(f"\nDescription: {md_prompt.description}")

## Configure Judge Settings

In [ ]:
# Custom settings for judging
settings = JudgeSettings(
    temperature=0.0,  # Deterministic for consistency
    max_tokens=2048,  # Enough for detailed judgments
    prompt_style='multi_dimensional',  # 5-dimension scoring
)

print(f"Judge Settings:")
print(f"  Temperature: {settings.temperature}")
print(f"  Max tokens: {settings.max_tokens}")
print(f"  Prompt style: {settings.prompt_style}")

## Create a Judge

Using the factory function:

In [ ]:
# Example: Create an Ollama judge (local model)
# This requires Ollama to be running locally
try:
    judge = create_judge(
        provider='ollama',
        model='llama3.1:8b',
        settings=settings,
    )
    print(f"Created Ollama judge with model: {judge.model}")
    ollama_available = True
except Exception as e:
    print(f"Ollama not available: {e}")
    print("Install Ollama from https://ollama.com and run: ollama pull llama3.1:8b")
    ollama_available = False

In [ ]:
# Alternative: Create an OpenAI judge
openai_key = os.getenv('OPENAI_API_KEY')
if openai_key:
    judge = create_judge(
        provider='openai',
        model='gpt-4o',
        settings=settings,
    )
    print(f"Created OpenAI judge with model: {judge.model}")
    judge_available = True
elif ollama_available:
    judge_available = True
else:
    print("No judge available - set OPENAI_API_KEY or install Ollama")
    judge_available = False

## Sample Data for Judging

In [ ]:
# Sample question and response to judge
sample_items = [
    {
        'question': 'Explain the concept of requirements traceability in systems engineering.',
        'expected_answer': '''Requirements traceability is the ability to track requirements throughout 
        the system development lifecycle. It links requirements to their sources (backward traceability) 
        and to subsequent development artifacts like design, implementation, and tests (forward traceability). 
        This ensures completeness, enables impact analysis, and supports verification and validation.''',
        'response': '''Requirements traceability means being able to follow requirements from start to 
        finish in a project. It helps you see where each requirement came from and make sure it's been 
        implemented properly. It's useful for understanding the impact of changes.''',
    },
    {
        'question': 'What is the purpose of a System Requirements Review (SRR)?',
        'expected_answer': '''The System Requirements Review (SRR) is a formal technical review that 
        verifies system requirements are complete, consistent, and properly allocated. It ensures 
        stakeholder needs are addressed and the requirements baseline is ready for design.''',
        'response': '''SRR is a meeting where you look at requirements. It happens before design 
        starts to make sure everyone agrees on what the system should do.''',
    },
]

print(f"Prepared {len(sample_items)} items for judging")

## Run Judging

In [ ]:
if judge_available:
    # Judge a single response
    result = judge.judge(
        question=sample_items[0]['question'],
        expected_answer=sample_items[0]['expected_answer'],
        response=sample_items[0]['response'],
    )
    
    print("Judgment Result:")
    print(json.dumps(result, indent=2))
else:
    print("Example judgment output:")
    example_result = {
        'parse_success': True,
        'scores': {
            'technical_accuracy': 14,
            'conceptual_understanding': 12,
            'completeness': 10,
            'clarity_organization': 15,
            'professional_relevance': 11,
        },
        'total_score': 62,
        'justification': 'The response shows basic understanding but lacks depth...',
    }
    print(json.dumps(example_result, indent=2))

In [ ]:
if judge_available:
    # Batch judging
    results = judge.batch_judge(sample_items, progress=True)
    
    print(f"\nJudged {len(results)} responses")
    for i, result in enumerate(results):
        total = result.get('total_score', 'N/A')
        print(f"  Item {i+1}: Total Score = {total}")

## Scoring Dimensions

The multi-dimensional rubric evaluates:

1. **Technical Accuracy** (0-20): Correctness of facts and concepts
2. **Conceptual Understanding** (0-20): Depth of understanding
3. **Completeness** (0-20): Coverage of key points
4. **Clarity & Organization** (0-20): Communication quality
5. **Professional Relevance** (0-20): Real-world applicability

**Total Score**: 0-100

## CLI Alternative

```bash
# Judge with Ollama (local)
metaeval judge responses.json --provider ollama --model llama3.1:8b

# Judge with OpenAI
metaeval judge responses.json --provider openai --model gpt-4o

# With custom settings
metaeval judge responses.json \
    --provider openai \
    --model gpt-4o \
    --temperature 0.1 \
    --max-tokens 4096 \
    --prompt chain_of_thought

# List available prompts
metaeval prompts judge
```